# 🏦 AI in Digital Finance & Fintech — Lab Notebook
### MIT PE × Quanskill Bootcamp · Lab B: Agentic Pipeline

---

| Thông tin | Chi tiết |
|-----------|----------|
| **Môi trường** | Google Colab / VS Code / Jupyter |
| **Python** | 3.10+ |
| **Lab A** | ML Credit Risk — RandomForest Budget Predictor |
| **Lab B** | AI Agent Pipeline — Retrieval → ML → Memory → LLM |
| **Khung lý thuyết** | DBS ALAN · bunq Finn · Agentic Pipeline (Slides 5, 7, 9, 11–14) |

---

### 📋 Cấu trúc notebook

```
PHẦN 0  — Cài đặt môi trường & tạo dữ liệu
PHẦN 1  — Core Modules (Knowledge Base · ML Engine · LLM Explainer)
PHẦN 2  — Lab A: ML Credit Risk & Budget Prediction
PHẦN 3  — Lab B: AI Agent Pipeline End-to-End
PHẦN 4  — Prompt Engineering (Zero-Shot · Role · Structured)
PHẦN 5  — Gradio UI (chạy trực tiếp trong Colab)
PHẦN 6  — Bài tập & Câu hỏi thảo luận
```

> 💡 **Cách dùng:** Chạy từng cell theo thứ tự từ trên xuống dưới.  
> Mỗi section có **lý thuyết** → **code** → **kết quả** → **bài tập**.


---
## PHẦN 0 — Cài đặt Môi trường

> **Slide 0 · Phase 0** — Bước đầu tiên luôn là cài đặt đúng thư viện.  
> Notebook tự phát hiện môi trường (Colab / Local) và cài đặt phù hợp.


In [2]:
# ─── Cell 0.1 — Kiểm tra môi trường ─────────────────────────────────────────
import sys, os

IN_COLAB = 'google.colab' in sys.modules
ENV_NAME = "Google Colab" if IN_COLAB else "Local / VS Code"

print(f"🖥️  Môi trường  : {ENV_NAME}")
print(f"🐍  Python      : {sys.version.split()[0]}")
print(f"📂  Working dir : {os.getcwd()}")


🖥️  Môi trường  : Local / VS Code
🐍  Python      : 3.11.14
📂  Working dir : /Users/macbook/Documents/PHYTHON/AI4Fintech/fintech_lab/Google_Colab


In [3]:
# ─── Cell 0.2 — Cài đặt thư viện ────────────────────────────────────────────
# Chạy cell này trước tiên — khoảng 60–90 giây lần đầu

import subprocess, sys

PACKAGES = [
    "scikit-learn>=1.4",
    "pandas>=2.0",
    "numpy>=1.26",
    "gradio>=4.44",
    "plotly>=5.22",
    "transformers>=4.40",   # FLAN-T5 (optional)
    "sentencepiece",         # tokenizer
    "tabulate",              # markdown tables
]

for pkg in PACKAGES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("✅ Tất cả thư viện đã cài xong!")
print("   scikit-learn · pandas · numpy · gradio · plotly · transformers")



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip

[notice] A new release of pip is available: 2

✅ Tất cả thư viện đã cài xong!
   scikit-learn · pandas · numpy · gradio · plotly · transformers


In [ ]:
# ─── Cell 0.3 — Tạo cấu trúc thư mục dự án ──────────────────────────────────
import os

BASE = "fintech_lab"
dirs = [BASE, f"{BASE}/core", f"{BASE}/data"]
for d in dirs:
    os.makedirs(d, exist_ok=True)

# Tạo __init__.py cho package
open(f"{BASE}/core/__init__.py", "w").close()

print("📁 Cấu trúc thư mục:")
for root, dirs_, files in os.walk(BASE):
    level = root.replace(BASE, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}📂 {os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  📄 {f}")


In [ ]:
# ─── Cell 0.4 — Tạo dữ liệu mẫu: Enterprise Knowledge Base ──────────────────
# Đây là 'bộ não' của AI Agent — Knowledge Base với các vấn đề doanh nghiệp

KB_CSV = """issue_pattern,solution,department,urgency_level,recommended_action
unauthorized financial variance detected in quarterly general ledger,Deploy Automated Reconciliation Agents to audit ledger transactions in real-time,Finance Advisor,high,Initiate immediate operational deep-dive audit and lock matching batch records
latency spike and data desynchronization on core cloud ERP platform,Upgrade API gateway bandwidth allocation and establish Redis distributed caching layer,IT Copilot,high,Schedule immediate out-of-hours infrastructure maintenance and cluster scaling
delays in calculating employees end-of-year performance KPIs and bonuses,Implement automated pipeline processing to stream real-time operational data into ERP,HR Bot,medium,Configure automated approvals workflows via central enterprise resource system
acute shortage of senior automated QA software engineers for deployment,Outsource mid-term software engineering tasks or invoke internal cross-skilling program,Product Management,medium,Open vendor bidding for certified engineering subcontractors to meet project milestones
slow monthly financial reporting consolidation,Deploy AI-powered financial close automation with real-time exception flagging,Finance Advisor,medium,Integrate automated reconciliation engine with ERP general ledger feeds
manual invoice reconciliation causing errors,Implement optical character recognition pipeline with automated 3-way matching,Finance Advisor,high,Deploy RPA bots for immediate invoice queue processing and error flagging
customer churn prediction failure,Rebuild churn model with ensemble approach including behavioral and transactional features,IT Copilot,medium,Conduct model performance audit and retrain with 24-month rolling dataset
loan approval processing bottleneck,Deploy ML-based credit scoring with automated decision engine and human-in-loop escalation,Finance Advisor,high,Implement tiered approval workflow with automated pre-screening for standard applications
regulatory compliance reporting delay,Automate regulatory data extraction with AI-powered report generation and validation,Finance Advisor,high,Establish automated regulatory calendar with pre-built reporting templates
vendor payment processing delays,Implement straight-through processing with AI-powered exception handling and approval routing,Finance Advisor,medium,Deploy automated payment orchestration with real-time status tracking
"""

with open("fintech_lab/data/enterprise_knowledge.csv", "w") as f:
    f.write(KB_CSV)

print("✅ enterprise_knowledge.csv — 10 patterns")
print()

# Đọc lại để xác nhận
import pandas as pd
kb_df = pd.read_csv("fintech_lab/data/enterprise_knowledge.csv")
print(kb_df[["issue_pattern","department","urgency_level"]].to_string(index=False))


In [ ]:
# ─── Cell 0.5 — Tạo dữ liệu mẫu: Project Cost Dataset ──────────────────────
# Dataset này dùng để train RandomForest Budget Prediction model
# Columns: team_size, duration_weeks, risk_level(1=low,2=med,3=high), region, estimated_cost

import numpy as np, pandas as pd, random

random.seed(42); np.random.seed(42)

REGION_MAP  = {"northeast":1.15, "northwest":1.10, "southeast":1.05, "southwest":1.00}
RISK_RATE   = {1: 0.05, 2: 0.15, 3: 0.30}
RATE_PER_WK = 800   # USD per person per week

rows = []
regions = list(REGION_MAP.keys())
for _ in range(120):
    ts  = random.randint(2, 60)
    dur = random.randint(2, 40)
    rl  = random.choice([1, 2, 3])
    reg = random.choice(regions)
    base   = ts * dur * RATE_PER_WK
    rr     = RISK_RATE[rl]
    oh_mul = REGION_MAP[reg]
    cost   = base * (1 + rr) * oh_mul * (1 + np.random.normal(0, 0.08))
    rows.append([ts, dur, rl, reg, round(max(cost, 1000), 2)])

cost_df = pd.DataFrame(rows, columns=["team_size","duration_weeks","risk_level","region","estimated_cost"])
cost_df.to_csv("fintech_lab/data/project_cost_data.csv", index=False)

print("✅ project_cost_data.csv — 120 records")
print()
print(cost_df.describe().round(0).to_string())


---
## PHẦN 1 — Core Modules: Xây dựng từng Layer của AI Agent Pipeline

> **Slides 11–13 · Agentic Pipeline Architecture**
>
> ```
> User Input → [Layer 1: KB Retrieval] → [Layer 2: ML Prediction] → [Layer 3: Memory] → [Layer 4: LLM Explainer]
> ```
>
> Chúng ta sẽ build từng layer một, rồi kết nối lại thành pipeline hoàn chỉnh.


### Layer 1 — Knowledge Base (TF-IDF Retrieval)

**Lý thuyết (Slide 5):** Khi user gửi vấn đề, AI Agent không hỏi LLM ngay mà  
**truy xuất knowledge base** bằng TF-IDF + cosine similarity để tìm pattern gần nhất.

```
User query → TF-IDF vectorize → cosine_similarity(query, KB) → top-k result
```

- `TF-IDF` (Term Frequency–Inverse Document Frequency): đo độ quan trọng của từ  
- `cosine_similarity`: đo góc giữa 2 vector — càng nhỏ càng giống nhau  
- Output: `department`, `urgency_level`, `confidence_score`, `recommended_action`


In [ ]:
# ─── Cell 1.1 — Knowledge Base Module ───────────────────────────────────────
%%writefile fintech_lab/core/database.py
"""
Layer 1 — Semantic Knowledge Base
TF-IDF + cosine similarity retrieval
Slide 5, 12, 13 — MIT PE × Quanskill Bootcamp
"""
import os, numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATA_PATH = os.path.join(os.path.dirname(__file__), "..", "data", "enterprise_knowledge.csv")

class EnterpriseKnowledgeBase:
    def __init__(self):
        self.df = pd.read_csv(DATA_PATH)
        self.vectorizer = TfidfVectorizer(ngram_range=(1,2), stop_words="english", max_features=5000)
        self.index_matrix = self.vectorizer.fit_transform(self.df["issue_pattern"].tolist())
        print(f"[KB] ✅ Loaded {len(self.df)} patterns — TF-IDF index ready")

    def search(self, query: str, top_k: int = 1) -> list:
        q_vec = self.vectorizer.transform([query])
        scores = cosine_similarity(q_vec, self.index_matrix)[0]
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [{
            "issue_pattern":    self.df.iloc[i]["issue_pattern"],
            "solution":         self.df.iloc[i]["solution"],
            "department":       self.df.iloc[i]["department"],
            "urgency_level":    self.df.iloc[i]["urgency_level"],
            "recommended_action": self.df.iloc[i]["recommended_action"],
            "confidence_score": float(scores[i]),
        } for i in top_indices]

    def get_stats(self) -> dict:
        return {
            "total_patterns":        len(self.df),
            "departments":           self.df["department"].nunique(),
            "urgency_distribution":  self.df["urgency_level"].value_counts().to_dict(),
        }


In [ ]:
# ─── Cell 1.2 — Test Knowledge Base ──────────────────────────────────────────
import sys; sys.path.insert(0, "fintech_lab")
from core.database import EnterpriseKnowledgeBase

kb = EnterpriseKnowledgeBase()

# Test với 3 queries khác nhau
test_queries = [
    "delays in calculating employees end-of-year performance KPIs and bonuses",
    "manual invoice reconciliation causing errors",
    "customer churn prediction failure",
]

print("\n" + "="*65)
print("KNOWLEDGE BASE — TF-IDF RETRIEVAL TEST")
print("="*65)
for q in test_queries:
    result = kb.search(q, top_k=1)[0]
    print(f"\n🔍 Query   : {q[:55]}...")
    print(f"   ✅ Match  : {result['issue_pattern'][:55]}...")
    print(f"   🏛  Dept   : {result['department']}")
    print(f"   🚨 Urgency: {result['urgency_level'].upper()}")
    print(f"   🎯 Score  : {result['confidence_score']:.1%}")

stats = kb.get_stats()
print(f"\n📊 KB Stats: {stats['total_patterns']} patterns | {stats['departments']} departments")
print(f"   Urgency distribution: {stats['urgency_distribution']}")


### Layer 2 — ML Budget Predictor (RandomForest)

**Lý thuyết (Slide 11–12 · Lab A):**  
RandomForest = tập hợp nhiều Decision Trees, mỗi cây vote độc lập → ensemble prediction.

**Công thức cost:**
```
Labor        = team_size × duration_weeks × $800/week
Infrastructure = base × 20%
Risk Buffer  = base × risk_rate  (low=5%, med=15%, high=30%)
Overhead     = base × region_multiplier  (SW=1.0x → NE=1.15x)
Total        = Labor + Infrastructure + Risk Buffer + Overhead
```

**95% Confidence Interval** = mean ± 1.96 × std (từ 200 individual trees)


In [ ]:
# ─── Cell 1.3 — ML Budget Predictor Module ───────────────────────────────────
%%writefile fintech_lab/core/predictor.py
"""
Layer 2-3 — Predictive Budget Engine
RandomForestRegressor: 200 trees, 95% CI via tree variance
Slide 11, 13 — DBS ALAN pattern
"""
import os, numpy as np, pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score

DATA_PATH = os.path.join(os.path.dirname(__file__), "..", "data", "project_cost_data.csv")
RISK_MAP  = {"low": 1, "medium": 2, "high": 3}
REGIONS   = ["northeast", "northwest", "southeast", "southwest"]

class BudgetPredictor:
    def __init__(self):
        df = pd.read_csv(DATA_PATH)
        self.le = LabelEncoder(); self.le.fit(REGIONS)
        X = np.column_stack([df["team_size"].values, df["duration_weeks"].values,
                             df["risk_level"].values, self.le.transform(df["region"])])
        y = df["estimated_cost"].values
        self.rf = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42)
        self.rf.fit(X, y)
        names = ["team_size","duration_weeks","risk_level","region"]
        self.feature_importances = dict(zip(names, self.rf.feature_importances_.tolist()))
        self.train_r2 = r2_score(y, self.rf.predict(X))
        print(f"[ML] ✅ RandomForest ready — R²={self.train_r2:.3f}  |  n_trees=200")

    def _encode(self, ts, dw, rl, reg):
        r   = RISK_MAP.get(rl.lower(), 2)
        reg = reg.lower() if reg.lower() in REGIONS else "southeast"
        return np.array([[ts, dw, r, self.le.transform([reg])[0]]])

    def predict(self, team_size, duration_weeks, risk_level, region) -> dict:
        X = self._encode(team_size, duration_weeks, risk_level, region)
        tree_preds = np.array([t.predict(X)[0] for t in self.rf.estimators_])
        mean, std  = float(np.mean(tree_preds)), float(np.std(tree_preds))
        return {"estimated_cost": round(mean,2),
                "confidence_low":  round(max(0, mean-1.96*std),2),
                "confidence_high": round(mean+1.96*std,2),
                "std_dev":         round(std,2),
                "model_accuracy":  round(self.train_r2*100,2)}

    def get_cost_breakdown(self, ts, dw, rl, reg) -> dict:
        total = self.predict(ts,dw,rl,reg)["estimated_cost"]
        rp    = 0.08 * RISK_MAP.get(rl.lower(),2) / 3
        return {"total": total, "labor": round(total*0.55,2),
                "infrastructure": round(total*0.20,2), "risk_buffer": round(total*rp,2),
                "overhead": round(total*(1-0.55-0.20-rp),2)}

    def sensitivity_analysis(self, base_ts, base_wk, base_rl, base_reg) -> list:
        out = []
        for ts in [max(1,base_ts-4), base_ts, base_ts+4, base_ts+8]:
            out.append({"variable":f"Team {ts}","cost":self.predict(ts,base_wk,base_rl,base_reg)["estimated_cost"],"category":"team_size"})
        for wk in [max(1,base_wk-2), base_wk, base_wk+2, base_wk+4]:
            out.append({"variable":f"{wk}w","cost":self.predict(base_ts,wk,base_rl,base_reg)["estimated_cost"],"category":"duration"})
        for rl in ["low","medium","high"]:
            out.append({"variable":rl.capitalize(),"cost":self.predict(base_ts,base_wk,rl,base_reg)["estimated_cost"],"category":"risk"})
        return out


In [ ]:
# ─── Cell 1.4 — Test ML Budget Predictor ─────────────────────────────────────
from core.predictor import BudgetPredictor

pred = BudgetPredictor()

# Base profile
base = pred.predict(12, 8, "medium", "southwest")

print("\n" + "="*55)
print("ML BUDGET PREDICTOR — BASE PROFILE")
print(f"  Team: 12 người | Duration: 8 tuần | Risk: Medium | Region: Southwest")
print("="*55)
print(f"  💰 Estimated Cost : ${base['estimated_cost']:>10,.2f}")
print(f"  📉 95% CI (low)   : ${base['confidence_low']:>10,.0f}")
print(f"  📈 95% CI (high)  : ${base['confidence_high']:>10,.0f}")
print(f"  📊 Model R²       : {base['model_accuracy']}%")

print("\n📐 Feature Importances (% contribution to prediction):")
for feat, imp in sorted(pred.feature_importances.items(), key=lambda x:-x[1]):
    bar = "█" * int(imp * 40)
    print(f"  {feat:<18} {bar} {imp:.1%}")

breakdown = pred.get_cost_breakdown(12, 8, "medium", "southwest")
print("\n💼 Cost Breakdown:")
for k,v in breakdown.items():
    if k != "total":
        pct = v/breakdown['total']*100
        print(f"  {k:<20} ${v:>9,.0f}   ({pct:.1f}%)")
print(f"  {'TOTAL':<20} ${breakdown['total']:>9,.0f}")


### Layer 3 — Executive Explainer (LLM / Template)

**Lý thuyết (Slide 7, 9 · bunq Finn pattern):**  
Raw ML output (numbers) → LLM → Human-friendly executive narrative.

Hai chế độ:
1. **Template mode** (mặc định): deterministic, nhanh, không cần GPU
2. **FLAN-T5 mode**: Small Language Model rewrite, cần RAM ~4GB


In [ ]:
# ─── Cell 1.5 — LLM Explainer Module ─────────────────────────────────────────
%%writefile fintech_lab/core/explainer.py
"""
Layer 4-5 — Executive Explainer
Role Prompting + Constrained Rewrite (Slide 7, 9)
bunq Finn pattern: raw ML output → human-friendly narrative
FLAN-T5: lazy load, optional
"""
import threading
from datetime import datetime

URGENCY_ICON = {"high":"🔴","medium":"🟡","low":"🟢"}
DEPT_ICON    = {"Finance Advisor":"💰","IT Copilot":"⚙️","HR Bot":"👥","Product Management":"📦"}

_lock=threading.Lock(); _tok=_mdl=None; _loaded=False; _err=None

def _load_flan():
    global _tok,_mdl,_loaded,_err
    if _loaded: return True
    try:
        from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
        print("[FLAN] Loading google/flan-t5-large (~60s first time)...")
        _tok = AutoTokenizer.from_pretrained("google/flan-t5-large")
        _mdl = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large")
        _loaded=True; print("[FLAN] ✅ Ready"); return True
    except Exception as e:
        _err=str(e); print(f"[FLAN] ❌ {e}"); return False

def _flan_rewrite(text):
    prompt = (
        "You are a senior compliance officer at a global bank. "
        "Rewrite the following operational alert as a professional executive brief. "
        "Rules: preserve all facts, no jargon, max 8 sentences, "
        "end with: 'Please consult a licensed professional before acting.'\n\nText:\n" + text
    )
    with _lock:
        inp = _tok(prompt, return_tensors="pt", truncation=True, max_length=512)
        out = _mdl.generate(**inp, max_new_tokens=280, temperature=0.3, do_sample=True)
        return _tok.decode(out[0], skip_special_tokens=True).strip()

class EnterpriseExplainer:
    def __init__(self): self.query_log=[]

    def build_structured(self, business_name, issue_desc, kb, br) -> str:
        dept=kb.get("department","Ops"); urg=kb.get("urgency_level","medium")
        ts=datetime.now().strftime("%Y-%m-%d %H:%M")
        report=(
            f"=== EXECUTIVE OPERATIONAL ANALYSIS REPORT ===\n"
            f"Business : {business_name}\nGenerated: {ts}\n{'─'*52}\n\n"
            f"{URGENCY_ICON.get(urg,'🟡')} INCIDENT CLASSIFICATION\n"
            f"- Issue      : {issue_desc}\n"
            f"- Agent      : {DEPT_ICON.get(dept,'🏢')} {dept}\n"
            f"- Alert      : {urg.upper()}\n"
            f"- Confidence : {kb.get('confidence_score',0)*100:.1f}%\n\n"
            f"RECOMMENDED DIGITAL SOLUTION\n{kb.get('solution','N/A')}\n\n"
            f"IMMEDIATE ACTION\n{kb.get('recommended_action','N/A')}\n\n"
            f"BUDGET PROJECTION (RandomForest ML Engine)\n"
            f"- Cost   : ${br.get('estimated_cost',0):,.2f} USD\n"
            f"- 95% CI : ${br.get('confidence_low',0):,.0f} – ${br.get('confidence_high',0):,.0f}\n"
            f"- R²     : {br.get('model_accuracy',0):.1f}%\n\n"
            f"{'─'*52}\n"
            f"Note: Auto-generated per MIT PE Digital Transformation Framework."
        )
        self.query_log.append({"ts":ts,"business":business_name,"dept":dept,
                               "urg":urg,"cost":br.get("estimated_cost",0)})
        return report

    def build_executive_brief(self, business_name, issue_desc, kb, br, use_llm=False):
        base=self.build_structured(business_name,issue_desc,kb,br)
        if not use_llm: return base,"FLAN-T5 not activated — deterministic report"
        if not _load_flan(): return base,f"⚠️ FLAN load failed: {_err}"
        try:
            r=_flan_rewrite(base)
            if r and len(r)>80:
                return (f"[✅ FLAN-T5-Large Enhanced]\n{'─'*52}\n{r}\n\n{'─'*52}\n[Deterministic]\n{base}",
                        "✅ FLAN-T5-Large rewrite applied")
            return base,"⚠️ FLAN output too short — fallback"
        except Exception as e:
            return base,f"⚠️ FLAN error: {e}"

    def format_api_log(self, kb, br) -> str:
        return (f"[Agent]     : {kb.get('department','N/A')}\n"
                f"[Pattern]   : '{kb.get('issue_pattern','N/A')}'\n"
                f"[Confidence]: {kb.get('confidence_score',0)*100:.2f}%\n"
                f"[Action]    : {kb.get('recommended_action','N/A')}\n"
                f"[Budget]    : ${br.get('estimated_cost',0):,.2f} USD")


In [ ]:
# ─── Cell 1.6 — Backend Orchestrator ─────────────────────────────────────────
%%writefile fintech_lab/app_backend.py
"""
Orchestrator — kết nối tất cả pipeline layers
"""
import sys, os; sys.path.insert(0, os.path.dirname(__file__))
from core.database  import EnterpriseKnowledgeBase
from core.predictor import BudgetPredictor
from core.explainer import EnterpriseExplainer

_kb=_pred=_exp=None
def _init():
    global _kb,_pred,_exp
    if _kb is None:
        _kb=EnterpriseKnowledgeBase()
        _pred=BudgetPredictor()
        _exp=EnterpriseExplainer()
    return _kb,_pred,_exp

def run_analysis(business_name, admin_email, team_size, duration_weeks,
                 risk_level, region, issue_description, use_llm=False):
    kb,pred,exp=_init()
    kb_results = kb.search(issue_description, top_k=1)
    kb_result  = kb_results[0] if kb_results else {
        "issue_pattern":issue_description,"solution":"Manual review.",
        "department":"Operations","urgency_level":"medium",
        "recommended_action":"Escalate.","confidence_score":0.0}
    budget_result  = pred.predict(team_size,duration_weeks,risk_level,region)
    cost_breakdown = pred.get_cost_breakdown(team_size,duration_weeks,risk_level,region)
    sensitivity    = pred.sensitivity_analysis(team_size,duration_weeks,risk_level,region)
    executive_brief, flan_status = exp.build_executive_brief(
        business_name,issue_description,kb_result,budget_result,use_llm)
    api_log = exp.format_api_log(kb_result,budget_result)
    return {"executive_brief":executive_brief,"flan_status":flan_status,
            "api_log":api_log,"kb_result":kb_result,"budget_result":budget_result,
            "cost_breakdown":cost_breakdown,"sensitivity":sensitivity,
            "feature_importances":pred.feature_importances,"kb_stats":kb.get_stats()}

def get_kb_stats():
    kb,_,_=_init(); return kb.get_stats()


In [ ]:
# ─── Cell 1.7 — Test Full Pipeline ───────────────────────────────────────────
import sys; sys.path.insert(0,"fintech_lab")
from app_backend import run_analysis

result = run_analysis(
    business_name     = "VinFintech Corp",
    admin_email       = "admin@vinfintech.vn",
    team_size         = 12,
    duration_weeks    = 8,
    risk_level        = "medium",
    region            = "southwest",
    issue_description = "delays in calculating employees end-of-year performance KPIs and bonuses",
    use_llm           = False
)

print(result["executive_brief"])
print("\n" + "─"*52)
print("API LOG:")
print(result["api_log"])


---
## PHẦN 2 — Lab A: ML Credit Risk & Budget Prediction

> **Slide 11 · Lab A — RandomForest Budget Engine**
>
> Lab A tập trung vào **ML phần**: train model, đánh giá, visualize.  
> Bạn sẽ hiểu rõ RandomForest hoạt động như thế nào trong bối cảnh Fintech.


In [ ]:
# ─── Cell 2.1 — Khám phá Dataset ────────────────────────────────────────────
import pandas as pd, numpy as np, matplotlib.pyplot as plt

df = pd.read_csv("fintech_lab/data/project_cost_data.csv")

print("📊 DATASET OVERVIEW")
print(f"   Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print()
print(df.describe().round(0))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Project Cost Dataset — Phân phối dữ liệu", fontsize=13, fontweight="bold")

# Distribution of estimated_cost
axes[0].hist(df["estimated_cost"], bins=20, color="#00B4D8", edgecolor="white")
axes[0].set_title("Estimated Cost ($)")
axes[0].set_xlabel("Cost"); axes[0].set_ylabel("Count")

# Team size vs cost
axes[1].scatter(df["team_size"], df["estimated_cost"], alpha=0.6, color="#8B5CF6", s=40)
axes[1].set_title("Team Size vs Cost")
axes[1].set_xlabel("Team Size"); axes[1].set_ylabel("Cost ($)")

# Risk level distribution
rl_counts = df["risk_level"].value_counts().sort_index()
rl_labels = {1:"Low",2:"Medium",3:"High"}
axes[2].bar([rl_labels[k] for k in rl_counts.index], rl_counts.values,
            color=["#10B981","#F59E0B","#EF4444"], edgecolor="white")
axes[2].set_title("Risk Level Distribution")
axes[2].set_xlabel("Risk Level"); axes[2].set_ylabel("Count")

plt.tight_layout(); plt.show()


In [ ]:
# ─── Cell 2.2 — Train RandomForest & Đánh giá ───────────────────────────────
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import LabelEncoder
import numpy as np, pandas as pd

df = pd.read_csv("fintech_lab/data/project_cost_data.csv")
le = LabelEncoder()
le.fit(["northeast","northwest","southeast","southwest"])

X = np.column_stack([
    df["team_size"].values,
    df["duration_weeks"].values,
    df["risk_level"].values,
    le.transform(df["region"])
])
y = df["estimated_cost"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
rf = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
r2  = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

# Cross-validation
cv_scores = cross_val_score(rf, X, y, cv=5, scoring="r2")

print("="*55)
print("RANDOMFOREST MODEL EVALUATION")
print("="*55)
print(f"  R² Score (test)       : {r2:.4f}  ({r2*100:.1f}%)")
print(f"  MAE (test)            : ${mae:,.0f}")
print(f"  Cross-val R² (5-fold) : {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print(f"  Training samples      : {len(X_train)}")
print(f"  Test samples          : {len(X_test)}")
print()

# Actual vs Predicted plot
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,6))
ax.scatter(y_test, y_pred, alpha=0.7, color="#1565C0", s=50)
mn,mx = min(y_test.min(),y_pred.min()), max(y_test.max(),y_pred.max())
ax.plot([mn,mx],[mn,mx],"r--",lw=1.5,label="Perfect prediction")
ax.set_xlabel("Actual Cost ($)", fontsize=12)
ax.set_ylabel("Predicted Cost ($)", fontsize=12)
ax.set_title(f"RandomForest: Actual vs Predicted  (R²={r2:.3f})", fontsize=13, fontweight="bold")
ax.legend(); plt.tight_layout(); plt.show()


In [ ]:
# ─── Cell 2.3 — Feature Importance ───────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

feature_names  = ["team_size", "duration_weeks", "risk_level", "region"]
importances    = rf.feature_importances_
sorted_idx     = np.argsort(importances)

colors = ["#00B4D8","#8B5CF6","#F59E0B","#10B981"]

fig, ax = plt.subplots(figsize=(8,5))
bars = ax.barh([feature_names[i] for i in sorted_idx],
               [importances[i]   for i in sorted_idx],
               color=[colors[i]  for i in sorted_idx], edgecolor="white", height=0.5)

for bar, val in zip(bars, [importances[i] for i in sorted_idx]):
    ax.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
            f"{val:.1%}", va="center", fontsize=11, fontweight="bold")

ax.set_title("Feature Importance — RandomForest Budget Model", fontsize=13, fontweight="bold")
ax.set_xlabel("Importance Score")
ax.set_xlim(0, max(importances)*1.25)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout(); plt.show()

print("\n📐 Interpretation:")
for i in sorted_idx[::-1]:
    print(f"  {feature_names[i]:<18} → {importances[i]:.1%} contribution")


In [ ]:
# ─── Cell 2.4 — Sensitivity Analysis ─────────────────────────────────────────
# Slide 11: Thay đổi từng feature để xem cost thay đổi như thế nào
import sys; sys.path.insert(0,"fintech_lab")
from app_backend import run_analysis
import pandas as pd, matplotlib.pyplot as plt

BASE = dict(team_size=12, duration_weeks=8, risk_level="medium", region="southwest")

def get_cost(**kw):
    p = {**BASE, **kw}
    r = run_analysis("Lab", "lab@test.com", p["team_size"], p["duration_weeks"],
                     p["risk_level"], p["region"], "KPI delays", False)
    return r["budget_result"]["estimated_cost"]

# Vary team size
teams = [5, 8, 12, 16, 20, 25, 30]
costs_team = [get_cost(team_size=t) for t in teams]

# Vary duration
durations = [4, 6, 8, 10, 12, 16, 20]
costs_dur  = [get_cost(duration_weeks=d) for d in durations]

# Vary risk
risks      = ["low","medium","high"]
costs_risk = [get_cost(risk_level=r) for r in risks]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Sensitivity Analysis — Cost vs Each Feature", fontsize=13, fontweight="bold")

axes[0].plot(teams, costs_team, "o-", color="#00B4D8", lw=2, ms=8)
axes[0].set_title("Team Size Impact")
axes[0].set_xlabel("Team Size (người)"); axes[0].set_ylabel("Est. Cost ($)")
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"${x:,.0f}"))

axes[1].plot(durations, costs_dur, "s-", color="#8B5CF6", lw=2, ms=8)
axes[1].set_title("Duration Impact")
axes[1].set_xlabel("Duration (tuần)")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"${x:,.0f}"))

colors_r = ["#10B981","#F59E0B","#EF4444"]
axes[2].bar(risks, costs_risk, color=colors_r, edgecolor="white", width=0.5)
axes[2].set_title("Risk Level Impact")
axes[2].set_xlabel("Risk Level")
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"${x:,.0f}"))
for i, (r, c) in enumerate(zip(risks, costs_risk)):
    axes[2].text(i, c+500, f"${c:,.0f}", ha="center", fontsize=10, fontweight="bold")

for ax in axes: ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout(); plt.show()

print("\n📊 Delta Analysis (Base: 12 team, 8 weeks, medium, SW):")
base_cost = get_cost()
print(f"  Base Cost                : ${base_cost:>10,.0f}")
print(f"  Team +8 (→20)            : ${get_cost(team_size=20):>10,.0f}  (Δ ${get_cost(team_size=20)-base_cost:+,.0f})")
print(f"  Duration +4 (→12w)       : ${get_cost(duration_weeks=12):>10,.0f}  (Δ ${get_cost(duration_weeks=12)-base_cost:+,.0f})")
print(f"  Risk High                : ${get_cost(risk_level='high'):>10,.0f}  (Δ ${get_cost(risk_level='high')-base_cost:+,.0f})")
print(f"  Region Northeast (×1.15) : ${get_cost(region='northeast'):>10,.0f}  (Δ ${get_cost(region='northeast')-base_cost:+,.0f})")


In [ ]:
# ─── Cell 2.5 — Cost Breakdown Pie Chart ─────────────────────────────────────
import sys; sys.path.insert(0,"fintech_lab")
from app_backend import run_analysis
import matplotlib.pyplot as plt

result = run_analysis("VinFintech", "demo@lab.vn", 12, 8, "medium", "southwest",
                      "delays in calculating employees end-of-year performance KPIs", False)
bd = result["cost_breakdown"]

labels  = ["Labor", "Infrastructure", "Risk Buffer", "Overhead"]
values  = [bd["labor"], bd["infrastructure"], bd["risk_buffer"], bd["overhead"]]
colors  = ["#1565C0", "#00B4D8", "#EF4444", "#F59E0B"]
explode = [0.04, 0.02, 0.02, 0.02]

fig, ax = plt.subplots(figsize=(7,7))
wedges, texts, autotexts = ax.pie(
    values, labels=labels, colors=colors, explode=explode,
    autopct=lambda p: f"${p/100*bd['total']:,.0f}\n({p:.1f}%)",
    startangle=90, pctdistance=0.75,
    wedgeprops=dict(edgecolor="white", linewidth=2)
)
for t in autotexts: t.set_fontsize(10); t.set_fontweight("bold")

centre = plt.Circle((0,0), 0.50, color="white")
ax.add_patch(centre)
ax.text(0,0.05, f"${bd['total']:,.0f}", ha="center", va="center",
        fontsize=14, fontweight="bold", color="#0A1628")
ax.text(0,-0.12, "Total Cost", ha="center", va="center",
        fontsize=10, color="#64748B")

ax.set_title("Cost Breakdown — RandomForest Budget Model", fontsize=13, fontweight="bold", pad=20)
plt.tight_layout(); plt.show()


### 🏋️ Lab A — Bài tập thực hành

> **Thực hiện từng bài, ghi kết quả vào ô bên dưới**

**Bài 1:** Thay `team_size=25`, `risk_level="high"`, `region="northeast"`. Budget thay đổi bao nhiêu %?

**Bài 2:** Vẽ lại Feature Importance chart nhưng dùng `n_estimators=50` thay vì 200. R² thay đổi không?

**Bài 3:** Tại sao team_size thường là feature quan trọng nhất? Giải thích bằng công thức.

**Bài 4:** Nếu thêm feature `seniority_level` (junior=1, mid=2, senior=3), bạn sẽ thêm vào đâu trong pipeline?


---
## PHẦN 3 — Lab B: AI Agent Pipeline End-to-End

> **Slides 5, 12–14 · Lab B — Agentic Pipeline**
>
> Lab B là phần nâng cao: kết nối tất cả layers thành một AI Agent thực sự.  
> So sánh Chatbot (User→LLM→Answer) vs Agent (Multi-layer pipeline).

```
                    ┌─────────────────────────────────────────────────┐
  User Input   →→→  │  KB Retrieval → ML Predict → Memory → Explainer │ →→→ Output
                    └─────────────────────────────────────────────────┘
                             Agentic Pipeline (Lab B)
```


In [ ]:
# ─── Cell 3.1 — Pipeline Demo: 5 Issue Types ────────────────────────────────
import sys; sys.path.insert(0,"fintech_lab")
from app_backend import run_analysis

issues = [
    ("delays in calculating employees end-of-year performance KPIs and bonuses", 12, 8, "medium", "southwest"),
    ("slow monthly financial reporting consolidation",                            15, 10, "high",   "northeast"),
    ("manual invoice reconciliation causing errors",                              8,  6, "high",   "southeast"),
    ("customer churn prediction failure",                                         10, 12, "medium", "northwest"),
    ("loan approval processing bottleneck",                                       20, 16, "high",   "northeast"),
]

print("="*75)
print(f"{'ISSUE':<42} {'DEPT':<18} {'URGENCY':<8} {'BUDGET'}")
print("="*75)

results = []
for issue, ts, dur, rl, reg in issues:
    r = run_analysis("TestCorp","test@lab.vn", ts, dur, rl, reg, issue, False)
    kb = r["kb_result"]; br = r["budget_result"]
    print(f"{issue[:40]:<42} {kb['department']:<18} {kb['urgency_level'].upper():<8} "
          f"${br['estimated_cost']:>10,.0f}")
    results.append(r)

print("="*75)
print("\n✅ All 5 issue types processed through pipeline successfully!")


In [ ]:
# ─── Cell 3.2 — Deep Dive: Xem chi tiết 1 pipeline run ─────────────────────
import sys; sys.path.insert(0,"fintech_lab")
from app_backend import run_analysis
import json

r = run_analysis(
    business_name     = "Techcombank Digital",
    admin_email       = "cto@techcombank.vn",
    team_size         = 15,
    duration_weeks    = 10,
    risk_level        = "high",
    region            = "northeast",
    issue_description = "manual invoice reconciliation causing errors",
    use_llm           = False
)

print("┌─────────────────────────────────────────────────────────────┐")
print("│              PIPELINE EXECUTION REPORT                      │")
print("└─────────────────────────────────────────────────────────────┘")
print()
print("📍 LAYER 1 — KB Retrieval (TF-IDF)")
print(f"   Matched Pattern : {r['kb_result']['issue_pattern']}")
print(f"   Department      : {r['kb_result']['department']}")
print(f"   Confidence      : {r['kb_result']['confidence_score']:.1%}")
print(f"   Urgency         : {r['kb_result']['urgency_level'].upper()}")
print()
print("📍 LAYER 2 — ML Budget Prediction (RandomForest)")
br = r['budget_result']
print(f"   Estimated Cost  : ${br['estimated_cost']:,.2f}")
print(f"   95% CI          : ${br['confidence_low']:,.0f} – ${br['confidence_high']:,.0f}")
print(f"   Model R²        : {br['model_accuracy']}%")
print()
print("📍 LAYER 3 — Memory (Session Log)")
print(f"   FLAN Status     : {r['flan_status']}")
print()
print("📍 LAYER 4 — API Log")
print(r['api_log'])
print()
print("📍 LAYER 5 — Executive Brief")
print(r['executive_brief'])


In [ ]:
# ─── Cell 3.3 — Session Memory: Nhiều queries liên tiếp ─────────────────────
import sys; sys.path.insert(0,"fintech_lab")
from app_backend import run_analysis
import pandas as pd

session_log = []

queries = [
    ("VinFintech Corp",   "delays in calculating employees KPIs", 12, 8,  "medium", "southwest"),
    ("Techcombank",       "manual invoice reconciliation errors",  15, 10, "high",   "northeast"),
    ("VPBank AI Division","customer churn prediction failure",     10, 12, "medium", "northwest"),
]

for business, issue, ts, dur, rl, reg in queries:
    r  = run_analysis(business,"demo@lab.vn",ts,dur,rl,reg,issue,False)
    kb = r["kb_result"]; br = r["budget_result"]
    session_log.append({
        "Business": business,
        "Issue": issue[:35]+"...",
        "Department": kb["department"],
        "Urgency": kb["urgency_level"].upper(),
        "Budget": f"${br['estimated_cost']:,.0f}",
        "Confidence": f"{kb['confidence_score']:.1%}",
    })

df_log = pd.DataFrame(session_log)
print("📋 SESSION MEMORY LOG — Agent nhớ tất cả queries trong session")
print()
print(df_log.to_string(index=False))
print()
print(f"💾 Total queries: {len(session_log)} | Avg budget: ${sum(float(r['Budget'][1:].replace(',','')) for r in session_log)/len(session_log):,.0f}")


---
## PHẦN 4 — Prompt Engineering: 3 Kỹ thuật Chính

> **Slide 7, 9 · bunq Finn Pattern · DBS ADA Pattern**
>
> Prompt Engineering không phải "hỏi AI bằng tiếng Anh" — đây là **kỹ thuật thiết kế**.  
> Lab này demo 3 kỹ thuật tăng dần về độ phức tạp:
>
> | Kỹ thuật | Mức độ | Use case |
> |----------|--------|----------|
> | Zero-Shot | Cơ bản | Tóm tắt nhanh |
> | Role Prompting | Trung cấp | Customer comms (bunq Finn) |
> | Structured Output | Nâng cao | API integration (DBS ADA) |


In [ ]:
# ─── Cell 4.1 — Setup: Lấy raw data từ pipeline ─────────────────────────────
import sys; sys.path.insert(0,"fintech_lab")
from app_backend import run_analysis

r  = run_analysis("VinFintech","demo@lab.vn",12,8,"medium","southwest",
                  "delays in calculating employees end-of-year performance KPIs and bonuses",False)
kb = r["kb_result"]; br = r["budget_result"]

RAW_DATA = f"""
Department: {kb['department']} | Urgency: {kb['urgency_level']}
Issue: delays in calculating KPIs and bonuses
Solution: {kb['solution']}
Action: {kb['recommended_action']}
Budget: ${br['estimated_cost']:,.0f} USD | Confidence: {kb['confidence_score']*100:.0f}%
"""

print("RAW OUTPUT FROM AI AGENT PIPELINE:")
print("─"*55)
print(RAW_DATA)
print("─"*55)
print("\n→ Dưới đây chúng ta sẽ transform data này bằng 3 kỹ thuật prompt khác nhau")


In [ ]:
# ─── Cell 4.2 — STEP 1: Zero-Shot Prompt (Slide 7) ──────────────────────────
# Kỹ thuật đơn giản nhất: không có context, không có role

ZERO_SHOT = f"""Summarize this operational data: {RAW_DATA}"""

print("="*60)
print("STEP 1 — ZERO-SHOT PROMPT")
print("="*60)
print("\nPROMPT:")
print("─"*40)
print(ZERO_SHOT)
print("─"*40)
print("""
→ NHẬN XÉT:
  • Không có role → AI chỉ paraphrase, không thêm giá trị
  • Output generic, technical, không phù hợp business communication
  • Đây là cách 90% người dùng LLM — KHÔNG nên dùng trong production
  
  ❌ Kết quả: "The operational data indicates a medium urgency HR issue..."
""")


In [ ]:
# ─── Cell 4.3 — STEP 2: Role Prompting (bunq Finn Pattern — Slide 7, 9) ────
# bunq Finn: LLM đóng vai senior compliance officer để rewrite thông báo

ROLE_PROMPT = f"""You are a senior compliance officer at a global bank.
Translate operational system alerts into clear, customer-friendly communications.

Rules:
- No technical jargon
- Maximum 3 sentences
- Include ONE specific next action
- Tone: professional but empathetic
- End with: "Contact support@bank.com if you have questions."

Operational Alert:
{RAW_DATA}

Write the customer advisory:"""

print("="*60)
print("STEP 2 — ROLE PROMPTING (bunq Finn Pattern)")
print("="*60)
print("\nPROMPT:")
print("─"*40)
print(ROLE_PROMPT)
print("─"*40)
print("""
→ NHẬN XÉT:
  • Role = compliance officer → output phù hợp banking context
  • Constraints (3 sentences, no jargon) → kiểm soát được format
  • "Contact support" → call-to-action rõ ràng
  
  ✅ Expected output: "Our system has detected delays in your KPI processing cycle.
     Our HR automation team is actively resolving this issue, with resolution 
     expected within 48 hours. Contact support@bank.com if you have questions."
     
  → Đây là CHÍNH XÁC cách bunq Finn hoạt động: raw alert → customer advisory
""")


In [ ]:
# ─── Cell 4.4 — STEP 3: Structured Output (DBS ADA Pattern — Slide 7) ───────
# DBS ADA: output JSON để feed vào workflow automation (JIRA, ServiceNow)

STRUCTURED_PROMPT = f"""Analyze this operational alert and respond ONLY in valid JSON.

Alert:
{RAW_DATA}

Required JSON structure:
{{
  "risk_score": "LOW|MEDIUM|HIGH",
  "rationale": "one sentence explaining the risk",
  "recommended_action": "specific next step for the operations team",
  "estimated_resolution_days": <integer>,
  "escalation_required": <boolean>,
  "budget_approved": <boolean>,
  "department_routing": "Finance Advisor|IT Copilot|HR Bot|Product Management"
}}

Respond with JSON only, no explanation:"""

print("="*60)
print("STEP 3 — STRUCTURED OUTPUT (DBS ADA Pattern)")
print("="*60)
print("\nPROMPT:")
print("─"*40)
print(STRUCTURED_PROMPT)
print("─"*40)
print("""
→ NHẬN XÉT:
  • JSON output → có thể auto-feed vào JIRA/ServiceNow API
  • "ONLY in valid JSON" → constrain output format nghiêm ngặt
  • Fields khớp với database schema → không cần parsing thêm
  
  ✅ Expected output:
  {
    "risk_score": "MEDIUM",
    "rationale": "KPI delays impact employee compensation and compliance",
    "recommended_action": "Deploy ERP automation pipeline within 48 hours",
    "estimated_resolution_days": 3,
    "escalation_required": false,
    "budget_approved": true,
    "department_routing": "HR Bot"
  }
  
  → Đây là pattern DBS ADA dùng để tích hợp AI vào enterprise workflow
""")


### 🏋️ Prompt Engineering — Bài tập thực hành

**Bài 1:** Viết Role Prompt cho một **Risk Manager** thay vì Compliance Officer.  
Kết quả có khác không? Tại sao?

**Bài 2:** Thêm field `"gdpr_compliant": boolean` vào Structured Output prompt.  
Đây liên quan đến GDPR Article 22 (quyền được giải thích) như thế nào?

**Bài 3:** Tại sao Structured Output (JSON) quan trọng hơn free-text trong enterprise integration?

**Bài 4 (nâng cao):** Kết hợp Role + Structured: viết prompt cho compliance officer  
output JSON với tone professional. Có thể làm không?


---
## PHẦN 5 — Gradio UI: Chạy Web App trong Colab

> **Slide 14 · Phase 6 — Gradio Frontend**
>
> Gradio tạo web interface tương tác không cần HTML/CSS/JS.  
> Trong Colab: `share=True` tạo public URL (có hiệu lực 72h).


In [ ]:
# ─── Cell 5.1 — Gradio App (Full UI) ─────────────────────────────────────────
import sys, os, json
sys.path.insert(0,"fintech_lab")
os.chdir("fintech_lab")   # đổi working dir để module imports hoạt động

import gradio as gr
import plotly.graph_objects as go
import pandas as pd, numpy as np
from app_backend import run_analysis

# ── Palettes & Options ──────────────────────────────────────────
PALETTES = {
    "Lab Guide":   ["#00B4D8","#1565C0","#8B5CF6","#10B981","#F59E0B"],
    "Risk Review": ["#EF4444","#F59E0B","#1565C0","#64748B","#10B981"],
    "Operations":  ["#10B981","#00B4D8","#1565C0","#F59E0B","#8B5CF6"],
}
BUSINESS_OPTIONS = [
    "VinFintech Corp","Techcombank Digital","VPBank AI Division",
    "MBBank Analytics","Sacombank Operations","ACB Risk Management",
    "HDBank Finance Dept","TPBank Innovation Lab","VietCredit Group","FE Credit Analytics",
]
ISSUE_OPTIONS = [
    "delays in calculating employees end-of-year performance KPIs and bonuses",
    "slow monthly financial reporting consolidation",
    "manual invoice reconciliation causing errors",
    "customer churn prediction failure",
    "loan approval processing bottleneck",
]
CHART_LIST = ["Cost Breakdown","Feature Importance","Budget Confidence","Urgency Distribution","Cost vs Team Size"]
session_history = []

# ── Chart helpers ──────────────────────────────────────────────
def empty_chart():
    fig = go.Figure()
    fig.add_annotation(text="⬆ Nhấn Analyze để xem biểu đồ",x=0.5,y=0.5,
                       xref="paper",yref="paper",showarrow=False,
                       font=dict(color="#94A3B8",size=14))
    fig.update_layout(height=340,paper_bgcolor="#fff",plot_bgcolor="#fff",
                      margin=dict(l=24,r=24,t=34,b=24),
                      xaxis=dict(visible=False),yaxis=dict(visible=False))
    return fig

def make_charts(r_json, theme, chart):
    if r_json == "{}": return empty_chart()
    r = json.loads(r_json); p = PALETTES[theme]
    if chart == "Cost Breakdown":
        bd = r["cost_breakdown"]
        fig = go.Figure(go.Pie(
            labels=["Labor","Infrastructure","Risk Buffer","Overhead"],
            values=[bd.get("labor",0),bd.get("infrastructure",0),bd.get("risk_buffer",0),bd.get("overhead",0)],
            hole=.45,marker_colors=p[:4]))
        fig.update_layout(title="Cost Breakdown",height=340,margin=dict(l=24,r=24,t=48,b=24))
    elif chart == "Feature Importance":
        fi = sorted(r["feature_importances"].items(),key=lambda x:x[1])
        fig = go.Figure(go.Bar(x=[v for _,v in fi],y=[k for k,_ in fi],
                               orientation="h",marker_color=p[0]))
        fig.update_layout(title="Feature Importance",height=340,
                          margin=dict(l=140,r=24,t=48,b=28),
                          plot_bgcolor="#fff",paper_bgcolor="#fff")
    elif chart == "Budget Confidence":
        br = r["budget_result"]
        cost,low,high = br["estimated_cost"],br["confidence_low"],br["confidence_high"]
        x = np.linspace(low,high,100)
        y = np.exp(-(x-cost)**2/(2*5000**2))
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=x,y=y,fill="tozeroy",line=dict(color=p[0],width=2.5)))
        fig.add_vline(x=cost,line_color=p[4],line_width=2)
        fig.update_layout(title="Budget Confidence",height=340,
                          margin=dict(l=48,r=24,t=48,b=38),
                          plot_bgcolor="#fff",paper_bgcolor="#fff")
    elif chart == "Urgency Distribution":
        d = r["kb_stats"]["urgency_distribution"]
        fig = go.Figure(go.Pie(labels=list(d.keys()),values=list(d.values()),hole=.6,marker_colors=p))
        fig.update_layout(title="Urgency Distribution",height=340,margin=dict(l=24,r=24,t=48,b=24))
    else:
        df = pd.read_csv("data/project_cost_data.csv")
        fig = go.Figure(go.Scatter(x=df["team_size"],y=df["estimated_cost"],
                                   mode="markers",marker=dict(color=p[0],size=9,opacity=0.75)))
        fig.update_layout(title="Cost vs Team Size",height=340,
                          margin=dict(l=54,r=24,t=48,b=42),
                          plot_bgcolor="#fff",paper_bgcolor="#fff")
    return fig

# ── Main analysis ───────────────────────────────────────────────
def analyze(business,email,issue,team,dur,risk,region,flan,chart,theme):
    r = run_analysis(business,email,int(team),int(dur),risk,region,issue,flan)
    br = r["budget_result"]; kb = r["kb_result"]
    budget  = f"${br['estimated_cost']:,.0f}"
    conf    = f"{kb['confidence_score']*100:.1f}%"
    dept    = kb['department']
    urg     = kb['urgency_level'].upper()
    ml_txt  = (f"Estimated Cost : {budget}\n"
               f"95% CI         : ${br['confidence_low']:,.0f} – ${br['confidence_high']:,.0f}\n"
               f"Model Accuracy : {br['model_accuracy']}%\nDepartment     : {dept}\n"
               f"Urgency        : {urg}\nConfidence     : {conf}")
    session_history.append({"Business":business,"Budget":budget,"Urgency":urg})
    rj = json.dumps(r,default=str)
    hist = pd.DataFrame(session_history).to_markdown(index=False) if session_history else "_No queries._"
    return (budget,conf,dept,urg,r["executive_brief"],ml_txt,
            r["api_log"],make_charts(rj,theme,chart),rj,hist,r["flan_status"])

# ── CSS ─────────────────────────────────────────────────────────
CSS = """
body{font-family:'DM Sans',sans-serif!important;background:#F8FAFF!important}
.gradio-container{max-width:1080px!important}
.hero{background:#0A1628;border-radius:0 0 14px 14px;padding:32px 28px 0;margin:-16px -8px 24px;position:relative}
.hero h1{color:#fff;font-size:2rem;margin:8px 0 6px}
.hero h1 span{color:#00B4D8}
.hero p{color:#A8C4E8;font-size:.9rem;margin:0 0 20px}
.badge{display:inline-block;background:rgba(0,180,216,.15);border:1px solid rgba(0,180,216,.42);
  border-radius:20px;padding:4px 14px;color:#00B4D8;font-size:11px;letter-spacing:2px;text-transform:uppercase}
.hbar{height:4px;background:rgba(255,255,255,.06);margin:0 -28px}
.hbar div{height:4px;background:linear-gradient(90deg,#00B4D8,#8B5CF6)}
.step{background:#fff;border:1px solid #E2E8F0;border-radius:12px;margin-bottom:16px;overflow:hidden}
.step-hd{display:flex;align-items:center;gap:12px;padding:14px 18px;border-bottom:1px solid #E2E8F0}
.step-num{width:34px;height:34px;border-radius:8px;display:flex;align-items:center;
  justify-content:center;font-family:monospace;font-size:13px;font-weight:700;flex-shrink:0}
.s1{background:#EEF2FF;color:#7C3AED}.s2{background:#ECFDF5;color:#059669}
.s3{background:rgba(0,180,216,.12);color:#0891B2}.s4{background:#FFF7ED;color:#EA580C}.s5{background:#F5F3FF;color:#7C3AED}
.step-hd h3{color:#0A1628;font-size:.95rem;margin:0}.step-hd p{color:#64748B;font-size:11px;margin:2px 0 0}
.sbadge{margin-left:auto;font-size:10px;font-weight:700;padding:2px 10px;border-radius:20px;text-transform:uppercase}
.t1{background:#EEF2FF;color:#7C3AED}.t2{background:#ECFDF5;color:#059669}
.t3{background:rgba(0,180,216,.1);color:#0891B2}.t4{background:#FFF7ED;color:#EA580C}
.step-bd{padding:18px 20px}
.callout{border-radius:8px;padding:10px 14px;margin:0 0 14px;display:flex;gap:10px;font-size:12.5px;line-height:1.5}
.callout.info{background:#EFF6FF;border-left:3px solid #1565C0}
.callout.tip{background:#ECFDF5;border-left:3px solid #10B981}
.callout.th{background:#F5F3FF;border-left:3px solid #8B5CF6}
.gradio-container button.primary{background:#0A1628!important;border:none!important;
  border-radius:10px!important;color:#fff!important;font-weight:700!important}
.mono-out textarea{background:#F8FAFF!important;color:#1E293B!important;
  font-family:'IBM Plex Mono',monospace!important;font-size:12px!important;line-height:1.7!important}
.api-out textarea{background:#F0F4FF!important;color:#0A1628!important;
  font-family:'IBM Plex Mono',monospace!important;font-size:12px!important;line-height:1.7!important}
"""

# ── UI Build ────────────────────────────────────────────────────
with gr.Blocks(title="AI Finance Lab — Colab",css=CSS) as demo:
    state = gr.State("{}")

    gr.HTML("""<div class='hero'>
      <div class='badge'>🎓 MIT PE × Quanskill · Lab B</div>
      <h1>AI in <span>Digital Finance</span> & Fintech</h1>
      <p>Agentic Pipeline: User Input → KB Retrieval → ML Prediction → Memory → LLM Explainer</p>
      <div class='hbar'><div></div></div></div>""")

    # STEP 1
    with gr.Group(elem_classes="step"):
        gr.HTML("""<div class='step-hd'>
          <div class='step-num s1'>01</div>
          <div><h3>Nhập Dữ Liệu Input</h3><p>Business context → Agent Pipeline · Slides 11–13</p></div>
          <span class='sbadge t1'>Setup</span></div>""")
        with gr.Column(elem_classes="step-bd"):
            gr.HTML("""<div class='callout th'>🧠 <div><b>Lý thuyết (Slide 11):</b>
              Các trường này = User Input layer. Issue → TF-IDF retrieval; Team/Duration/Risk/Region → ML features.</div></div>""")
            with gr.Row():
                bname = gr.Dropdown(BUSINESS_OPTIONS,value="VinFintech Corp",allow_custom_value=True,
                                    label="🏢 Business Name / Phòng ban")
                email = gr.Textbox(value="admin@vinfintech.vn",label="📧 Administrator Email")
            issue = gr.Dropdown(ISSUE_OPTIONS,value=ISSUE_OPTIONS[0],label="🔍 Issue Description")
            with gr.Row():
                team = gr.Slider(1,100,value=12,step=1,label="👥 Team Size (người)",
                                 info="Công thức: Labor = Team × Duration × $800/tuần")
                dur  = gr.Slider(1,52,value=8,step=1,label="📅 Duration (tuần)",
                                 info="1–4w Quick fix | 5–12w Project | 13w+ Programme")
            with gr.Row():
                risk   = gr.Radio(["low","medium","high"],value="medium",label="⚠️ Risk Level",
                                  info="Low=5% buffer | Medium=15% | High=30%")
                region = gr.Dropdown(["northeast","northwest","southeast","southwest"],value="southwest",
                                     label="🗺️ Region",info="SW=×1.0 | SE=×1.05 | NW=×1.10 | NE=×1.15 overhead")
            with gr.Row():
                flan    = gr.Checkbox(label="✨ Bật FLAN-T5 Explainer (cần RAM ~4GB)")
                flan_st = gr.Textbox(label="FLAN Status",interactive=False,scale=3)

    # STEP 2 — RUN
    with gr.Group(elem_classes="step"):
        gr.HTML("""<div class='step-hd'>
          <div class='step-num s2'>02</div>
          <div><h3>Chạy Phân Tích</h3><p>Kích hoạt toàn bộ Agentic Pipeline</p></div>
          <span class='sbadge t2'>Run</span></div>""")
        with gr.Column(elem_classes="step-bd"):
            gr.HTML("""<div class='callout info'>🚀 <div>
              Nhấn <b>▶ Analyze</b>: (1) TF-IDF KB search → (2) RandomForest predict →
              (3) Session memory write → (4) Executive brief generation</div></div>""")
            btn = gr.Button("▶  Analyze — Chạy AI Agent Pipeline",variant="primary",size="lg")

    # STEP 3 — KPI
    with gr.Group(elem_classes="step"):
        gr.HTML("""<div class='step-hd'>
          <div class='step-num s3'>03</div>
          <div><h3>KPI Kết Quả</h3><p>Budget · Confidence · Department · Urgency</p></div>
          <span class='sbadge t3'>Result</span></div>""")
        with gr.Column(elem_classes="step-bd"):
            with gr.Row():
                k1 = gr.Textbox(label="💰 Estimated Budget",value="Awaiting...",interactive=False)
                k2 = gr.Textbox(label="🎯 KB Confidence",  value="Awaiting...",interactive=False)
                k3 = gr.Textbox(label="🏛 Department",      value="Awaiting...",interactive=False)
                k4 = gr.Textbox(label="🚨 Urgency Level",  value="Awaiting...",interactive=False)

    # STEP 4 — DETAIL
    with gr.Group(elem_classes="step"):
        gr.HTML("""<div class='step-hd'>
          <div class='step-num s4'>04</div>
          <div><h3>Phân Tích Chi Tiết</h3><p>Executive Brief · ML Engine · API Log</p></div>
          <span class='sbadge t4'>Output</span></div>""")
        with gr.Column(elem_classes="step-bd"):
            with gr.Tabs():
                with gr.Tab("📋 Executive Summary"):
                    brief = gr.Textbox(label="Executive Brief",lines=12,interactive=False)
                with gr.Tab("🤖 ML Predictive Engine"):
                    gr.HTML("""<div class='callout th' style='margin:8px 0'>🔬 <div>
                      <b>RandomForest (Slide 12):</b> Cost = Labor + Infrastructure + Risk Buffer + Overhead</div></div>""")
                    ml_out = gr.Textbox(label="Budget Model Output",lines=10,
                                        interactive=False,elem_classes="mono-out")
                with gr.Tab("🔌 API Pipeline Log"):
                    gr.HTML("""<div class='callout info' style='margin:8px 0'>🔗 <div>
                      <b>Pipeline Log (Slide 14):</b> KB retrieval → ML prediction → Memory → LLM call</div></div>""")
                    log_out = gr.Textbox(label="Pipeline Log",lines=14,
                                         interactive=False,elem_classes="api-out")

    # STEP 5 — CHART
    with gr.Group(elem_classes="step"):
        gr.HTML("""<div class='step-hd'>
          <div class='step-num s5'>05</div>
          <div><h3>Analytics Dashboard</h3><p>Cost Breakdown · Feature Importance · Confidence · KB Stats</p></div>
          <span class='sbadge t1'>Charts</span></div>""")
        with gr.Column(elem_classes="step-bd"):
            with gr.Row():
                chart_dd = gr.Dropdown(CHART_LIST,value=CHART_LIST[0],label="📊 Chọn biểu đồ",scale=3)
                theme_dd = gr.Dropdown(list(PALETTES.keys()),value="Lab Guide",label="🎨 Palette",scale=1)
            chart_out = gr.Plot(value=empty_chart())

    # History
    with gr.Accordion("📂 Session History",open=False):
        hist_md = gr.Markdown("_Chưa có phân tích._")

    # ── Wiring ──
    inputs  = [bname,email,issue,team,dur,risk,region,flan,chart_dd,theme_dd]
    outputs = [k1,k2,k3,k4,brief,ml_out,log_out,chart_out,state,hist_md,flan_st]
    btn.click(fn=analyze,inputs=inputs,outputs=outputs)
    chart_dd.change(fn=make_charts,inputs=[state,theme_dd,chart_dd],outputs=chart_out)
    theme_dd.change(fn=make_charts,inputs=[state,theme_dd,chart_dd],outputs=chart_out)

# ── Launch ──────────────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules
demo.launch(share=IN_COLAB,   # public URL trong Colab, local trong VS Code
            server_name="0.0.0.0" if IN_COLAB else "127.0.0.1",
            server_port=7861,
            quiet=True)

print("✅ App đang chạy!")
if IN_COLAB:
    print("→ Colab: dùng URL public ở trên (valid 72h)")
else:
    print("→ Local: http://127.0.0.1:7861")


---
## PHẦN 6 — Bài tập Tổng hợp & Câu hỏi Thảo luận

> **Lab A + Lab B combined — Final Assessment**

---

### 🔬 Bài tập nâng cao

**Bài 1 — Extend Knowledge Base:**  
Thêm 3 issue patterns mới vào `enterprise_knowledge.csv` liên quan đến:
- Crypto payment fraud detection
- ESG reporting automation  
- Real-time FX hedging alerts

Chạy lại và xem KB confidence score thay đổi không?

**Bài 2 — Model Comparison:**  
Thay `RandomForestRegressor` bằng `GradientBoostingRegressor` (sklearn).  
R² tốt hơn hay kém hơn? Feature importance có thay đổi không?

**Bài 3 — API Integration Design:**  
Design một FastAPI endpoint `/analyze` nhận JSON input và return output từ `run_analysis()`.  
Vẽ sơ đồ request/response flow.

**Bài 4 — GDPR & Explainability:**  
GDPR Article 22 yêu cầu "right to explanation" cho automated decisions.  
- RandomForest có thể "explain" prediction không? Tại sao/tại sao không?
- SHAP values giải quyết vấn đề này như thế nào?

---

### 💬 Câu hỏi thảo luận nhóm

1. **Chatbot vs AI Agent:** Tại sao KPI delay issue cần AI Agent (multi-layer) thay vì chatbot (1 layer)?

2. **bunq Finn pattern:** Tại sao Role Prompting quan trọng trong banking communication? GDPR Article 22 ảnh hưởng thế nào?

3. **DBS ALAN:** Nếu thêm FAISS vector database thay TF-IDF, recall sẽ tốt hơn không? Trade-off là gì?

4. **Business case:** Nếu budget model sai 20%, risk cho tổ chức tài chính là gì? Làm thế nào giảm?

5. **Production readiness:** Pipeline này cần thêm gì để deploy lên production banking system?

---

### 📊 Chạy cell này để xem Summary Report


In [ ]:
# ─── Cell 6.1 — Final Summary Report ────────────────────────────────────────
import sys; sys.path.insert(0,".")
from app_backend import run_analysis, get_kb_stats

print("=" * 65)
print("   AI FINANCE LAB — COMPLETION SUMMARY")
print("=" * 65)

# Run final analysis
r = run_analysis("VinFintech Corp","student@lab.vn",12,8,"medium","southwest",
                 "delays in calculating employees end-of-year performance KPIs",False)

kb = r["kb_result"]; br = r["budget_result"]
stats = r["kb_stats"]

print(f"""
📋 PIPELINE STATUS:
   ✅ Layer 1 — Knowledge Base Retrieval  : {stats['total_patterns']} patterns indexed
   ✅ Layer 2 — RandomForest ML Engine    : R²={br['model_accuracy']}% accuracy  
   ✅ Layer 3 — Memory (Session Log)      : Active
   ✅ Layer 4 — Executive Explainer       : Template mode (FLAN-T5 optional)
   ✅ Layer 5 — Gradio UI                 : Deployed (check Cell 5.1)

📊 SAMPLE ANALYSIS RESULT:
   Issue       : {r['kb_result']['issue_pattern'][:50]}...
   Department  : {kb['department']}
   Urgency     : {kb['urgency_level'].upper()}
   Confidence  : {kb['confidence_score']:.1%}
   Budget      : ${br['estimated_cost']:,.0f} USD
   95% CI      : ${br['confidence_low']:,.0f} – ${br['confidence_high']:,.0f}

📚 CURRICULUM COVERAGE:
   ✅ Slide 5  — Chatbot vs AI Agent architecture
   ✅ Slide 7  — Prompt Engineering (Zero-Shot, Role, Structured)
   ✅ Slide 9  — bunq Finn constrained rewrite pattern
   ✅ Slide 11 — Lab A: RandomForest budget engine
   ✅ Slide 12 — DBS ALAN pipeline architecture
   ✅ Slide 13 — Memory + LLM Explainer layer
   ✅ Slide 14 — FastAPI + Gradio frontend

🏁 Lab hoàn thành! Bấm Run All → Analyze → Khám phá dashboard.
""")
print("=" * 65)
